# Notebook 6: Extending SGET

Practical pointers for common modifications. Use this as a reference when you need to add a feature.

## Prerequisites
- Notebooks 1-5 completed (understand the architecture, model, layout, and GUI)

## 1. Common Modifications

**Adding a new property to the property panel**:
Edit `views/property_panel.py` → `_show_node()`. Add a widget for your property (QLineEdit, QSpinBox, etc.), store it in `self._widgets`, and handle it in `_on_apply()`. The key mapping `"pos" → "center"` is where widget names translate to Neo4j property names. Also add the new property name to `ALLOWED_PROPERTIES` in `neo4j_crud.py` (whitelist for Cypher injection prevention).

**Adding a new menu action**:
Edit `main_window.py` → `_setup_menus()`. Add a `QAction` to the appropriate menu and connect it to a handler method. See `_add_node`, `_group_selected`, `_draw_region`, `_focus_on_subtree`, and `_export_image` for examples.

**Adding a new dialog**:
Follow the pattern in `widgets/add_node_dialog.py` or `widgets/group_dialog.py`:
1. Subclass `QDialog` with form fields
2. Accept/reject buttons via `QDialogButtonBox`
3. A `get_result()` or `execute_*()` method that returns data on accept, None on cancel
4. Call from `main_window.py` handler

**Adding a new layer**:
1. Add a `LayerStyle` entry to `LAYER_STYLES` in `utils/colors.py` — note that `category_chars` is a tuple of all known NodeSymbol prefixes (e.g., `("P", "t")` for MeshPlaces)
2. Add a CREATE template function in `neo4j_crud.py` and register it in `_CREATE_DISPATCH`
3. Add the layer to `_PARENT_LAYERS` in `widgets/group_dialog.py` if it participates in grouping
4. Everything else (model, layout, layer panel, add node dialog) picks it up automatically
5. Use `heracles.constants` for the label string — don't hardcode

**Adding a new node/edge item type**:
`NodeItem` and `EdgeItem` live in `views/graph_items.py`. Subclass or modify these to change how nodes/edges are rendered (color, shape, labels, etc.).

**Adding a new boundary visualization**:
Add a helper function in `utils/boundary.py` following the pattern of `make_radii_polygon_overlay()` or `make_bbox_overlay()`. Then add a case in `graph_view.py` → `_draw_boundaries()` to call it for the appropriate node type.

**Adding a new view or panel**:
Follow the pattern in `widgets/snapshot_panel.py`:
1. Create a widget class that takes the `SceneGraphModel` in its constructor
2. Connect to the model's signals (`graph_loaded`, `selection_changed`, `node_added`, etc.)
3. Add it as a dock widget in `main_window.py`
4. Add to the View menu via `view_menu.addAction(dock.toggleViewAction())`

**Adding a new interaction mode** (like the polygon tool):
1. Add state tracking in `GraphView` (e.g., `_polygon_mode_active`)
2. Override mouse events in `_ZoomableGraphicsView` to route to the mode
3. Emit a signal when the mode completes (e.g., `polygon_completed`)
4. Connect the signal in `main_window.py` to handle the result

## 2. Key Subsystems

**Generic node handling (heracles)**:
heracles uses property-presence to store whatever attributes a node has, plus an `attr_type` string for reconstruction. The `ATTR_TYPE_REGISTRY` in `graph_interface.py` maps type names to spark_dsg constructors. New attribute types are supported automatically if they follow the same property-presence pattern.

**Subtree queries**:
`model.get_descendants(node_symbol)` does BFS over CONTAINS edges and returns all transitive children. Used by the Focus on Subtree feature.

**NodeSymbol generation**:
`_next_node_symbol(model, layer_label)` in `widgets/add_node_dialog.py` scans existing nodes across ALL `category_chars` for the layer to find the next available index.

**Snapshots**:
The `SnapshotPanel` stores JSON files in `.sget_snapshots/` next to the loaded file. An "initial_load" snapshot is auto-saved on first load. Each snapshot is a full scene graph export via `model.save_to_json()`. The "Include mesh data" checkbox copies the mesh from the source DSG file into the snapshot, making it self-contained. Restoring a snapshot calls `model.load_from_json()`.

**Mesh file pointer**:
The source DSG file path is stored as a `_GraphMetadata` node in Neo4j. `model.get_source_file_path()` retrieves it. Used by snapshot mesh inclusion.

**Chat agent integration**:
The chat agent (heracles_agents) connects to the same Neo4j database. After the agent modifies the graph, press Ctrl+Shift+R in SGET to refresh. Custom agent tools can be added by registering them in heracles_agents' tool registry. Config files are in `config/`. Updated prompt drafts are in `docs/prompt_drafts/`.

## 3. Security

`neo4j_crud.py` validates property names against `ALLOWED_PROPERTIES` before building dynamic Cypher SET clauses. This prevents injection via user-supplied property names. When adding new properties to the schema, add them to this whitelist. The `attr_type` property is stored on every node but is not user-editable through the property panel.

Neo4j credentials default to environment variables (`HERACLES_NEO4J_USERNAME`, `HERACLES_NEO4J_PASSWORD`) with CLI fallback.

## 4. Running Tests

```bash
pytest                                  # All 47 SGET tests (requires Neo4j)
pytest tests/test_neo4j_crud.py         # CRUD layer (23 tests)
pytest tests/test_scene_graph_model.py  # Model tests (24 tests)
pytest -k "selection"                   # Tests matching a keyword
```

Heracles also has round-trip tests:
```bash
cd ~/software/mit/sget/heracles
python -m pytest heracles/tests/test_roundtrip.py  # 10 round-trip tests
```

All tests run against a live Neo4j instance on `localhost:7687`. There are currently no automated GUI tests — all tests cover the backend (CRUD and model).

## Summary

You now have the practical knowledge to extend SGET:
- **New properties**: add to `_show_node()` + `ALLOWED_PROPERTIES`
- **New dialogs**: follow `QDialog` + `get_result()` pattern
- **New layers**: add to `LAYER_STYLES` (with `category_chars` tuple) + `_CREATE_DISPATCH` + `_PARENT_LAYERS`
- **New item types**: modify `NodeItem`/`EdgeItem` in `views/graph_items.py`
- **New boundaries**: add helper in `utils/boundary.py`, call from `_draw_boundaries()`
- **New panels**: follow `SnapshotPanel` pattern (model signals + dock widget)
- **New modes**: state tracking + mouse event routing + signal on completion
- **Generic node handling**: heracles auto-stores any attribute type with `attr_type` dispatch
- **Subtree queries**: `model.get_descendants()` for transitive CONTAINS traversal
- **Mesh snapshots**: `save_to_json(include_mesh=True)` copies mesh from source
- **Security**: all property names validated against `ALLOWED_PROPERTIES` whitelist

For the full implementation details, see the source files directly — they are thoroughly documented with design-choice comments.